# 03_baseline_tabnet_optuna
TabNet + Optunaによるハイパーパラメータ最適化

In [1]:
%load_ext autoreload
%autoreload 2
import datetime, os, sys
from pathlib import Path
import numpy as np
import pandas as pd
PROJECT_ROOT = Path(
    "/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026"
)
sys.path.append(str(PROJECT_ROOT))
from common.tabnet.tabnet_model import run_tabnet
from common.tabnet.tabnet_model_optuna import run_tabnet_optuna
from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything
SEED = 42
seed_everything(seed=SEED)
TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

In [2]:
SCRIPT_NAME = "03_baseline_tabnet_optuna"
TODAY = datetime.datetime.now().strftime("%Y%m%d")
LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}.csv"

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

[2026-08-05 11:32:31] [INFO] === [03_baseline_tabnet_optuna] 実験開始 ===


In [3]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"
train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

In [4]:
def transform_monthly_to_wide_all(monthly_df):
    val_cols = [col for col in monthly_df.columns if col not in ["社員ID", "経過月数"]]
    monthly_wide = monthly_df.pivot(index="社員ID", columns="経過月数", values=val_cols)
    monthly_wide.columns = [f"{col}_m{month}" for col, month in monthly_wide.columns]
    return monthly_wide.reset_index()
train_monthly_wide = transform_monthly_to_wide_all(train_monthly)
test_monthly_wide = transform_monthly_to_wide_all(test_monthly)
train_df = pd.merge(train_persona, train_monthly_wide, on="社員ID", how="left")
test_df = pd.merge(test_persona, test_monthly_wide, on="社員ID", how="left")

In [5]:
def build_features(train, test, target_col, id_col):
    train_proc, test_proc = train.copy(), test.copy()
    non_num_cols = train_proc.select_dtypes(include=["object"]).columns.tolist()
    if id_col in non_num_cols:
        non_num_cols.remove(id_col)
    train_proc = train_proc.drop(columns=non_num_cols, errors="ignore")
    test_proc = test_proc.drop(columns=non_num_cols, errors="ignore")
    X_train = train_proc.drop(columns=[target_col, id_col], errors="ignore")
    y_train = train_proc[target_col]
    X_test = test_proc.drop(columns=[id_col], errors="ignore")
    return {"X_train": X_train, "y_train": y_train, "X_test": X_test}, test_proc[id_col]
input_data, test_ids = build_features(train_df, test_df, TARGET_COL, ID_COL)

In [6]:
tabnet_opt_params = {
    "n_splits": 5,
    "seed": SEED,
    "save_dir": str(SAVED_MODELS_DIR),
    "max_epochs": 30,
    "patience": 10,
    "batch_size": 256,
    "n_trials": 3,
}
logger.info("--- Optuna TabNet チューニング開始 ---")
opt_tabnet_res, best_tabnet_params = run_tabnet_optuna(data=input_data, params=tabnet_opt_params)
logger.info(f"TabNet Optuna Best Score: {opt_tabnet_res['best_score']:.4f}")

[2026-08-05 11:32:31] [INFO] --- Optuna TabNet チューニング開始 ---
Stop training because you reached max_epochs = 30 with best_epoch = 27 and best_val_0_logloss = 0.74142


/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Stop training because you reached max_epochs = 30 with best_epoch = 29 and best_val_0_logloss = 0.79538


/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Stop training because you reached max_epochs = 30 with best_epoch = 29 and best_val_0_logloss = 0.77448


...

/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Stop training because you reached max_epochs = 30 with best_epoch = 29 and best_val_0_logloss = 0.73366


/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Stop training because you reached max_epochs = 30 with best_epoch = 21 and best_val_0_logloss = 0.77678
[2026-08-05 11:33:47] [INFO] TabNet Optuna Best Score: 0.6807


/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [7]:
logger.info("--- TabNet 本学習実行 ---")
# Optunaメタパラメータと訓練時パラメータを除外
final_params = {k: v for k, v in best_tabnet_params.items() 
                if k not in ["n_trials", "max_epochs", "patience", "lr"]}
# lrはoptimizer_paramsとして再構築（Optunaで最適化されている場合）
if "lr" in best_tabnet_params:
    final_params["optimizer_params"] = {"lr": best_tabnet_params["lr"]}
final_params["max_epochs"] = 100
final_params["patience"] = 10
tabnet_res, _ = run_tabnet(data=input_data, params=final_params)
tabnet_cv = calculate_logloss(input_data["y_train"], tabnet_res["oof_preds"])
logger.info(f"TabNet CV Score: {tabnet_cv:.4f}")

[2026-08-05 11:33:47] [INFO] --- TabNet 本学習実行 ---

Early stopping occurred at epoch 68 with best_epoch = 58 and best_val_0_logloss = 0.66726


/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 29 with best_epoch = 19 and best_val_0_logloss = 0.67382


/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 71 with best_epoch = 61 and best_val_0_logloss = 0.65272


/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 53 with best_epoch = 43 and best_val_0_logloss = 0.6644


/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 50 with best_epoch = 40 and best_val_0_logloss = 0.68766
Successfully saved model at /Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/saved_models/20260805/03_baseline_tabnet_optuna/tabnet_model_fold0.zip
Successfully saved model at /Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/saved_models/20260805/03_baseline_tabnet_optuna/tabnet_model_fold1.zip
Successfully saved model at /Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/saved_models/20260805/03_baseline_tabnet_optuna/tabnet_model_fold2.zip
Successfully saved model at /Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/saved_models/20260805/03_baseline_tabnet_optuna/tabnet_model_fold3.zip
Successfully saved model at /Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/saved_models/2026080

/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [8]:
sub = pd.DataFrame({ID_COL: test_ids, TARGET_COL: tabnet_res["test_preds"]})
sub.to_csv(SUBMISSION_PATH, index=False)
logger.info(f"提出ファイル保存: {SUBMISSION_PATH}")
logger.info("=== 実験完了 ===")
print(f"\nOptuna Best: {opt_tabnet_res['best_score']:.4f}")
print(f"Final CV: {tabnet_cv:.4f}")

[2026-08-05 11:34:11] [INFO] 提出ファイル保存: /Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/data/output/20260805/20260805_03_baseline_tabnet_optuna.csv
[2026-08-05 11:34:11] [INFO] === 実験完了 ===

Optuna Best: 0.6807
Final CV: 0.6692
